In [253]:
import cv2
import numpy as np
from collections import deque
from datetime import datetime
import time

input_path = r"C:\Users\mpoko\Workspaces\Motion-Detection\src\input-files\WildLifeExample.mp4"
output_path = r"C:\Users\mpoko\Workspaces\Motion-Detection\src\output-files\Output"

In [254]:
## The captured video

vid = cv2.VideoCapture(input_path)

# Check if the video was opened successfully
if not vid.isOpened():
    print("Error: Could not open video file.")
else:
    print("Video file opened successfully!")

# Read the first frame1 to confirm reading
ret1, frame1 = vid.read()
ret2,  frame2 = vid.read()

fW = 440 
fH = 280
fA = fW * fH
 
if ret1:
    frame1 = cv2.resize(frame1, (fW, fH), fx = 0, fy = 0,
                         interpolation = cv2.INTER_CUBIC)

if ret2: 
    frame1 = cv2.resize(frame2, (fW, fH), fx = 0, fy = 0,
                         interpolation = cv2.INTER_CUBIC)
else:
    print("Error: Could not read the frame1.")

detection_zones = []

fps = vid.get(cv2.CAP_PROP_FPS)

fgbg = cv2.createBackgroundSubtractorMOG2()

frame_detected = [False ]* int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
len(frame_detected)



Video file opened successfully!


179

In [255]:
start = time.time()
# Buffer: allow motion to "persist" for N seconds after last detection
buffer_seconds = 0.01
buffer_frames = int(buffer_seconds * fps)
last_frame_detection = -10**9  # very small so first diff is huge

while True:
    ret, frame = vid.read()
    if not ret:
        break
    current_frame = int(vid.get(cv2.CAP_PROP_POS_FRAMES))
    idx = current_frame - 1  # convert to 0-based index
    frame = cv2.resize(frame, (fW, fH), fx = 0, fy = 0,
                         interpolation = cv2.INTER_LINEAR)
    fgmask = cv2.resize(fgmask, (fW, fH), fx = 0, fy = 0,
                         interpolation = cv2.INTER_LINEAR)
    fgmask = fgbg.apply(frame)

    contours, _ = cv2.findContours(
        fgmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    motion_detected = False
    for contour in contours:
        if cv2.contourArea(contour) < 1000:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        area = w*h
        ## Accept resonable sizes
        if fA * 0.01 < area < fA * 0.5:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
            with open("motion_log.txt","a") as f:
                f.write(f"Frame {current_frame}: Width {w}, Height {h}, Area {w*h}, Frame Sixe {frame.size}"+ "\n")
            motion_detected = True
    # buffer logic in frames
    if motion_detected:
        last_frame_detection = current_frame

    within_buffer = (current_frame - last_frame_detection) <= buffer_frames
    ## Last frame the buffer if it drops for a couple seconds 
    if motion_detected :
        current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        frame_detected[idx] = True
        timestamp_seconds = current_frame / fps
        detection_zones.append(timestamp_seconds)

        cv2.putText(
            frame1,
            f"Motion Detected: {current_timestamp}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2,
            cv2.LINE_AA
        )
    

    cv2.imshow("Background Subtraction", frame)
    cv2.imshow("Foreground Mask", fgmask)

    if cv2.waitKey(30) == 27:
        break
vid.release
cv2.destroyAllWindows()
end = time.time()

length = end - start

mins,second = divmod(length,60)

print(f"Video length: {int(mins)}:{second:.2f}")

Video length: 0:8.27


In [256]:
motion_segments = []

if detection_zones:
    segment_start = detection_zones[0]
    prev_time = detection_zones[0]

    gap_threshold = 1 / fps * 2  # allow small frame gaps

    for t in detection_zones[1:]:
        if t - prev_time > gap_threshold:
            # gap detected so we close segment
            motion_segments.append((segment_start, prev_time))
            segment_start = t

        prev_time = t

    # close final segment
    motion_segments.append((segment_start, prev_time))

In [257]:
for i, (start, end) in enumerate(motion_segments, 1):
    s_m, s_s = divmod(start, 60)
    e_m, e_s = divmod(end, 60)
    length = end - start
    l_m, l_s = divmod(length, 60)

    print(f"Segment {i}")
    print(f"  Start : {int(s_m)}:{s_s:05.2f}")
    print(f"  End   : {int(e_m)}:{e_s:05.2f}")
    print(f"  Length: {int(l_m)}:{l_s:05.2f}")
    print()
    cap = cv2.VideoCapture(input_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Calculate start/end frames
    start_frame = int(s_s * fps)
    end_frame = int(e_s * fps)

    fourcc = cv2.VideoWriter_fourcc(*'avc1') # Codec for mp4
    out = cv2.VideoWriter(f"{output_path}_{i}.mp4", fourcc, fps, (frame_width, frame_height))

    # Set starting position
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # Read and write frames
    current_frame = start_frame
    while current_frame < end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        
        out.write(frame)
        current_frame += 1
        
    cap.release()
    out.release()
    print(f"Clip saved: {output_path}")

Segment 1
  Start : 0:01.13
  End   : 0:04.90
  Length: 0:03.77

Clip saved: C:\Users\mpoko\Workspaces\Motion-Detection\src\output-files\Output


In [258]:
first_detection = detection_zones[0]
last_detection = detection_zones.pop()

# These are already video-relative timestamps
f_mins, f_sec = divmod(first_detection, 60)
l_mins, l_sec = divmod(last_detection, 60)

length = last_detection - first_detection
mins, seconds = divmod(length, 60)

print(f"Movement start: {int(f_mins)}:{f_sec:05.2f}")
print(f"Movement end: {int(l_mins)}:{l_sec:05.2f}")
print(f"Movement length: {int(mins)}:{seconds:05.2f}")

Movement start: 0:01.13
Movement end: 0:04.90
Movement length: 0:03.77
